In [2]:
import pandas as pd
import numpy as np

In [76]:
df = pd.read_csv('example_forecast-2.csv')

In [5]:
df.head()

,sku_id,subfamily,family,customer,forecast
0,SKU1001,cheese,dairy,costco,42
1,SKU1002,milk,dairy,walmart,19
2,SKU1003,yogurt,dairy,kroger,37
3,SKU1004,apples,fruits,costco,23
4,SKU1005,oranges,fruits,walmart,48


In [28]:
df[(df.customer== 'costco') & (df.family == 'dairy')]

,sku_id,subfamily,family,customer,forecast
0,SKU1001,cheese,dairy,costco,42
22,SKU1023,cream,dairy,costco,18
42,SKU1043,yogurt,dairy,costco,33
62,SKU1063,sour cream,dairy,costco,36
82,SKU1083,whipped cream,dairy,costco,46
100,SKU1101,cheese,dairy,costco,31
122,SKU1123,cream,dairy,costco,50
142,SKU1143,yogurt,dairy,costco,47
162,SKU1163,sour cream,dairy,costco,42
182,SKU1183,whipped cream,dairy,costco,49


In [29]:
tmp = df[(df.customer== 'costco') & (df.family == 'dairy')]

In [30]:
tmp['forecast'].sum()

np.int64(482)

In [31]:
multiplier = 1000 / 482

In [32]:
tmp['forecast']  = (tmp['forecast'] * multiplier).round(2)

/var/folders/vd/69kmbq6s0s331vf09zfxgmh40000gn/T/ipykernel_90061/2005682316.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp['forecast']  = (tmp['forecast'] * multiplier).round(2)


In [33]:
tmp

,sku_id,subfamily,family,customer,forecast
0,SKU1001,cheese,dairy,costco,87.14
22,SKU1023,cream,dairy,costco,37.34
42,SKU1043,yogurt,dairy,costco,68.46
62,SKU1063,sour cream,dairy,costco,74.69
82,SKU1083,whipped cream,dairy,costco,95.44
100,SKU1101,cheese,dairy,costco,64.32
122,SKU1123,cream,dairy,costco,103.73
142,SKU1143,yogurt,dairy,costco,97.51
162,SKU1163,sour cream,dairy,costco,87.14
182,SKU1183,whipped cream,dairy,costco,101.66


In [6]:
len(df)

300

In [ ]:
df['sku_id']

In [34]:
example_1 = [
    {"id": 1, "customer": "walmart", "level": "total", "new_forecast": 100},
    {
        "id": 2,
        "customer": "costco",
        "level": "family",
        "level_id": "dairy",
        "new_forecast": 50,
    },
]

example_2 = [
    {"id": 1, "customer": "costco", "level": "family", "level_id": "dairy", "new_forecast": 100},
    {
        "id": 2,
        "customer": "costco",
        "level": "sku",
        "level_id": "SKU1001",
        "new_forecast": 10,
    },
]

In [35]:
example_1

[{'id': 1, 'customer': 'walmart', 'level': 'total', 'new_forecast': 100},
 {'id': 2,
  'customer': 'costco',
  'level': 'family',
  'level_id': 'dairy',
  'new_forecast': 50}]

In [90]:
update = example_2[1]

In [91]:
updates = [update]

In [92]:
exclude_tracker = set()
# udpate SKU-level forecasts
for update in updates:
    if update['level'] == 'sku':
        tmp = df[(df.customer==update['customer']) & (df.sku_id == update['level_id'])]
        filtered_idx, exclude_tracker = idx_filter(list(tmp.index), exclude_tracker)
        

In [93]:
filtered_idx

[0]

In [38]:
if update['level'] == 'total':
    tmp = df[df.customer==update['customer']]
elif update['level'] == 'family':
    tmp = df[(df.customer==update['customer']) & (df.family == update['level_id'])]
elif update['level'] == 'subfamily':
    tmp = df[(df.customer==update['customer']) & (df.subfamily == update['level_id'])]
else: # SKU level
    tmp = df[(df.customer==update['customer']) & (df.sku_id == update['level_id'])]

{1}

In [85]:
def idx_filter(idx, exclude_tracker):
    res = []
    for i in idx:
        if i not in exclude_tracker:
            res.append(i)
            exclude_tracker.add(i)
    return res, exclude_tracker

In [40]:
current_forecast = tmp['forecast'].sum()

In [41]:
current_forecast

np.int64(2016)

In [46]:
multiplier = (update['new_forecast'] / current_forecast)

In [ ]:
15 skus under costco/dairy family -> 100 (482)
1 SKU1001 (costco) 10 (5)
1. update SKU1001 10-> 5
2. remaining new forecast 100 - 10 = 90
3. remaining original forecast 482 - 5 = 477
4. multiplier 90 / 477
5. apply multipler on the rest 14 items


In [47]:
multiplier

np.float64(0.0496031746031746)

In [53]:
for idx in tmp.index:
    df.loc[idx, 'forecast'] = df.loc[idx, 'forecast'] * multiplier

/var/folders/vd/69kmbq6s0s331vf09zfxgmh40000gn/T/ipykernel_90061/1924870213.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9424603174603174' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'forecast'] = df.loc[idx, 'forecast'] * multiplier


In [55]:
df

,sku_id,subfamily,family,customer,forecast
0,SKU1001,cheese,dairy,costco,42.000000
1,SKU1002,milk,dairy,walmart,0.942460
2,SKU1003,yogurt,dairy,kroger,37.000000
3,SKU1004,apples,fruits,costco,23.000000
4,SKU1005,oranges,fruits,walmart,2.380952
...,...,...,...,...,...
295,SKU1296,sourdough,bakery,walmart,1.636905
296,SKU1297,mussels,seafood,kroger,26.000000
297,SKU1298,sushi,seafood,publix,41.000000
298,SKU1299,kombucha,beverages,costco,17.000000


In [54]:
df[df.customer==update['customer']]

,sku_id,subfamily,family,customer,forecast
1,SKU1002,milk,dairy,walmart,0.942460
4,SKU1005,oranges,fruits,walmart,2.380952
7,SKU1008,beef,meat,walmart,1.339286
11,SKU1012,onions,vegetables,walmart,1.041667
15,SKU1016,tuna,seafood,walmart,1.636905
...,...,...,...,...,...
283,SKU1284,ricotta,dairy,walmart,1.686508
287,SKU1288,watermelon,fruits,walmart,1.537698
291,SKU1292,mushrooms,vegetables,walmart,1.835317
295,SKU1296,sourdough,bakery,walmart,1.636905


In [69]:
def forecast_update(df, updates=None):
    if updates is None or len(updates) == 0:
        return df

    for update in updates:
        if update['level'] == 'total':
            tmp = df[df.customer==update['customer']]
        elif update['level'] == 'family':
            tmp = df[(df.customer==update['customer']) & (df.family == update['level_id'])]
        elif update['level'] == 'subfamily':
            tmp = df[(df.customer==update['customer']) & (df.subfamily == update['level_id'])]
        else: # SKU level
            tmp = df[(df.customer==update['customer']) & (df.sku_id == update['level_id'])]

        # getting multiplier
        current_forecast = tmp['forecast'].sum()
        multiplier = (update['new_forecast'] / current_forecast)

        # update the df
        for idx in tmp.index:
            df.loc[idx, 'forecast'] = df.loc[idx, 'forecast'] * multiplier


    return df

In [70]:
updated_df = forecast_update(df, example_1)

In [71]:
updated_df

,sku_id,subfamily,family,customer,forecast
0,SKU1001,cheese,dairy,costco,4.936501
1,SKU1002,milk,dairy,walmart,0.942460
2,SKU1003,yogurt,dairy,kroger,37.000000
3,SKU1004,apples,fruits,costco,23.000000
4,SKU1005,oranges,fruits,walmart,2.380952
...,...,...,...,...,...
295,SKU1296,sourdough,bakery,walmart,1.636905
296,SKU1297,mussels,seafood,kroger,26.000000
297,SKU1298,sushi,seafood,publix,41.000000
298,SKU1299,kombucha,beverages,costco,17.000000


In [72]:
updated_df[updated_df.customer=='walmart']['forecast'].sum()

np.float64(100.0)

In [73]:
updated_df[(updated_df.customer=='costco') & (updated_df.family == 'dairy')]['forecast'].sum()

np.float64(49.99999999999999)

In [77]:
updated_df2 = forecast_update(df, example_2) 

/var/folders/vd/69kmbq6s0s331vf09zfxgmh40000gn/T/ipykernel_90061/2551067008.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '8.713692946058092' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'forecast'] = df.loc[idx, 'forecast'] * multiplier


In [78]:
updated_df2[(updated_df2.customer=='costco') & (updated_df2.family == 'dairy')]['forecast'].sum()

np.float64(101.28630705394191)